- Plot PCA using mean vector

In [7]:
!export PROJ="/project/md748/hz54/kav/KAV"

from utils.settings import *
from utils.files import find_all_pkl, get_sampled_sigma, read_pkl
from utils.ProcessRankLog import *
from utils.ProcessHeatmapData import *
from utils.EvaHeatmap import *
from utils.algo import *
import pandas as pd
import os

[1732313378.183758] [n0038:2371790:0]           ib_md.c:1232 UCX  WARN  IB: ibv_fork_init() was disabled or failed, yet a fork() has been issued.
[1732313378.183771] [n0038:2371790:0]           ib_md.c:1233 UCX  WARN  IB: data corruption might occur when using registered memory.


In [ ]:
from utils.files import read_pkl
from utils.settings import GAUSSIAN_LINEAR_FILE, STEER_LAYERS, CONCEPTS

import os
file = os.path.expandvars(GAUSSIAN_LINEAR_FILE)
mean, std = read_pkl(file)

from utils.EvaPCA import get_layer_pca
layer_mean = {}
for l in STEER_LAYERS:
    layer_mean[l] = []
    for c in CONCEPTS:
        layer_mean[l].append(mean[c][l])
    get_layer_pca(l, layer_mean[l], "rebuttal_mean")

In [ ]:
o_file = os.path.join(PARA_DIR, 'observed-linear-1000.pkl')
s_file = os.path.join(PARA_DIR, 'sampled-linear-1-sigma-1000.pkl')
sigma = get_sampled_sigma(s_file)
output_file = os.path.join(PARA_DIR, f"rebuttal-sim-mat-{sigma}-sigma-{OBSERVED_NUM}.pkl")
desc = 'rebuttal-sim-mat'
r_o_weights = new_layer_dict()
r_s_weights = new_layer_dict()
o_weights, s_weights = o_weights, s_weights = load_all_weights(o_file, s_file)
for l in LAYERS:
    r_o_weights[l], r_s_weights[l] = [], []
    for c in CONCEPTS:
        if c in o_weights.keys() and l in o_weights[c].keys():
            r_o_weights[l].append(copy.deepcopy(o_weights[c][l]))
        if c in s_weights.keys() and l in s_weights[c].keys():
            r_s_weights[l].append(copy.deepcopy(s_weights[c][l]))
        else:
            print(f"Lack weight vectors for concepts {c} at layer {l}")

2024-11-21 10:38:21,657 - Rank 0 - root - INFO - Loading weights...


In [24]:
a = mvs[0]
a = a - np.mean(a, axis=0, keepdims=True)
b = mvs[1]
b = b - np.mean(b, axis=0, keepdims=True)

similarity = np.dot(a.flatten(), b.flatten()) / (np.linalg.norm(a) * np.linalg.norm(b))
print(similarity)

0.000996577321559757


In [25]:
a = mvs[0]
b = mvs[1]
similarity = np.dot(a.flatten(), b.flatten()) / (np.linalg.norm(a) * np.linalg.norm(b))
print(similarity)

0.4507108142090511


In [30]:
a = np.array([[2,3,1], [1, 0, 2]])
b = np.array([[5,6,4], [4, 3, 2]])
similarity = cosine_similarity(a, b).flatten()
print(f"Cosine similarity: {sum(similarity)/len(similarity)}")

similarity = np.dot(a.flatten(), b.flatten()) / (np.linalg.norm(a) * np.linalg.norm(b))
print(f"Flattened cosine similarity: {similarity}")

a = a - np.mean(a, axis=0, keepdims=True)
b = b - np.mean(b, axis=0, keepdims=True)
similarity = np.dot(a.flatten(), b.flatten()) / (np.linalg.norm(a) * np.linalg.norm(b))
print(f"Centered cosine similarity: {similarity}")

Cosine similarity: 0.8111228002920732
Flattened cosine similarity: 0.8913130356431728
Centered cosine similarity: 0.6446583712203042


In [23]:
import numpy as np

# Create example vectors with shape (3,5)
v1 = np.array([[1, 2, 3, 4, 5],
              [6, 7, 8, 9, 10],
              [11, 12, 13, 14, 15]])

v2 = np.array([[2, 3, 4, 5, 6],
              [7, 8, 9, 10, 11], 
              [12, 13, 14, 15, 16]])

# Center the vectors
v1_centered = v1 - np.mean(v1)
v2_centered = v2 - np.mean(v2)
print(v1_centered)
print(v2_centered)

# Calculate similarity
similarity = np.dot(v1_centered.flatten(), v2_centered.flatten()) / (np.linalg.norm(v1_centered) * np.linalg.norm(v2_centered))

print(f"Centered cosine similarity: {similarity}")

[[-7. -6. -5. -4. -3.]
 [-2. -1.  0.  1.  2.]
 [ 3.  4.  5.  6.  7.]]
[[-7. -6. -5. -4. -3.]
 [-2. -1.  0.  1.  2.]
 [ 3.  4.  5.  6.  7.]]
Centered cosine similarity: 1.0


Compute the similarity score for 1 sigma and 5 sigma vectors

## Check the similarity between 1 sigma and 5 sigma vectors

In [31]:
one_sigma_file = os.path.join(PARA_DIR, "sampled-linear-1-sigma-1000.pkl")
five_sigma_file = os.path.join(PARA_DIR, "sampled-linear-5-sigma-1000.pkl")
one_sigma = read_pkl(one_sigma_file)
five_sigma = read_pkl(five_sigma_file)

In [40]:
for c in CONCEPTS:
    of_similarity_all_layers = []
    oo_similarity_all_layers = []
    for l in LAYERS:
        similarity = cosine_similarity(one_sigma[c][l], five_sigma[c][l]).flatten()
        similarity = sum(similarity)/len(similarity)
        of_similarity_all_layers.append(similarity)

        similarity = cosine_similarity(one_sigma[c][l], one_sigma[c][l]).flatten()
        similarity = sum(similarity)/len(similarity)
        oo_similarity_all_layers.append(similarity)
        
    print(f"Averaged simiarlity between 1-sigma and 5-sigma sampled vectors with concept {c} is:\n {pd.Series(of_similarity_all_layers).describe()}, {pd.Series(oo_similarity_all_layers).describe()}")

Averaged simiarlity between 1-sigma and 5-sigma sampled vectors with concept Cycling competition is:
 count    30.000000
mean      0.623372
std       0.062937
min       0.481742
25%       0.585827
50%       0.643216
75%       0.676670
max       0.696038
dtype: float64, count    30.000000
mean      0.943803
std       0.018107
min       0.897033
25%       0.935525
50%       0.950648
75%       0.958173
max       0.962045
dtype: float64
Averaged simiarlity between 1-sigma and 5-sigma sampled vectors with concept Football match is:
 count    30.000000
mean      0.780049
std       0.109624
min       0.530240
25%       0.713499
50%       0.755675
75%       0.890406
max       0.929673
dtype: float64, count    30.000000
mean      0.973325
std       0.018839
min       0.916998
25%       0.965406
50%       0.972675
75%       0.990155
max       0.994043
dtype: float64
Averaged simiarlity between 1-sigma and 5-sigma sampled vectors with concept Tennis tournament is:
 count    30.000000
mean      0.